In [0]:

# =========================================================
# PHASE 3: SILVER LAYER
# STEP 1: DATA QUALITY PROFILING
# =========================================================

from pyspark.sql import functions as F

BRONZE_TABLE = (
    "workspace.video_game_sales.bronze_video_game_sales"
)

# Read Bronze table
bronze_df = spark.table(BRONZE_TABLE)

total_records = bronze_df.count()

print("TOTAL BRONZE RECORDS:", total_records)

# 1. NULL VALUE ANALYSIS
null_expressions = [
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in bronze_df.columns
]

null_report = bronze_df.agg(*null_expressions)

print("\n--- NULL VALUE REPORT ---")
display(null_report)

# 2. DUPLICATE ANALYSIS
print("\n--- DUPLICATE ANALYSIS ---")

full_duplicates = (
    total_records - bronze_df.dropDuplicates().count()
)

duplicate_ranks = (
    total_records - bronze_df.select("Rank").distinct().count()
)

print("Full duplicate rows:", full_duplicates)
print("Duplicate Rank values:", duplicate_ranks)

# 3. YEAR ANALYSIS
print("\n--- YEAR DISTRIBUTION ---")

bronze_df.groupBy("Year") \
    .count() \
    .orderBy("Year") \
    .show(50, truncate=False)

# 4. SALES VALIDATION
sales_columns = [
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales"
]

print("\n--- SALES VALIDATION ---")

sales_report = bronze_df.agg(
    *[
        F.sum(
            F.when(F.col(c) < 0, 1).otherwise(0)
        ).alias(f"{c}_negative")
        for c in sales_columns
    ]
)

display(sales_report)

# 5. MISSING DIMENSION VALUES
dimension_columns = [
    "Name",
    "Platform",
    "Year",
    "Genre",
    "Publisher"
]

print("\n--- MISSING DIMENSIONS ---")

dimension_report = bronze_df.agg(
    *[
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in dimension_columns
    ]
)

display(dimension_report)

# 6. BUSINESS KEY CHECK
print("\n--- RANK UNIQUENESS ---")

bronze_df.groupBy("Rank") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(20, truncate=False)

print("\nDATA QUALITY PROFILING COMPLETED")


In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.video_game_sales.bronze_video_game_sales"
)

for column in bronze_df.columns:
    null_count = bronze_df.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column}: {null_count}")

In [0]:
total = bronze_df.count()

full_duplicates = total - bronze_df.dropDuplicates().count()

duplicate_ranks = (
    total - bronze_df.select("Rank").distinct().count()
)

print("Total records:", total)
print("Full duplicates:", full_duplicates)
print("Duplicate ranks:", duplicate_ranks)

In [0]:
bronze_df.groupBy("Year") \
    .count() \
    .orderBy("Year") \
    .show(50, truncate=False)

In [0]:
sales_columns = [
    "NA_Sales", "EU_Sales",
    "JP_Sales", "Other_Sales",
    "Global_Sales"
]

for column in sales_columns:
    print(
        column,
        "Negative values:",
        bronze_df.filter(F.col(column) < 0).count()
    )

In [0]:
dimensions = [
    "Name", "Platform", "Year",
    "Genre", "Publisher"
]

for column in dimensions:
    missing = bronze_df.filter(
        F.col(column).isNull() |
        (F.trim(F.col(column)) == "")
    ).count()

    print(f"{column}: {missing}")

In [0]:

# =========================================================
# SILVER: ADDITIONAL DATA QUALITY INVESTIGATION
# =========================================================

from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.video_game_sales.bronze_video_game_sales"
)

# 1. YEAR VALIDATION

year_profile = bronze_df.withColumn(
    "year_numeric",
    F.when(
        F.col("Year").rlike(r"^\d{4}$"),
        F.col("Year").cast("int")
    )
).withColumn(
    "year_status",
    F.when(
        F.col("year_numeric").isNotNull(),
        "VALID"
    ).otherwise("MISSING_OR_INVALID")
)

print("--- YEAR QUALITY ---")

year_profile.groupBy("year_status") \
    .count() \
    .show()

print("Invalid year values:")

year_profile.filter(
    F.col("year_status") == "MISSING_OR_INVALID"
).groupBy("Year").count().show()

# 2. WHITESPACE VALIDATION

text_columns = [
    "Name", "Platform", "Genre", "Publisher"
]

print("--- WHITESPACE CHECK ---")

for column in text_columns:
    count = bronze_df.filter(
        F.col(column) != F.trim(F.col(column))
    ).count()

    print(f"{column}: {count}")

# 3. REGIONAL VS GLOBAL SALES RECONCILIATION

sales_df = bronze_df.withColumn(
    "regional_sales_total",
    F.round(
        F.col("NA_Sales") +
        F.col("EU_Sales") +
        F.col("JP_Sales") +
        F.col("Other_Sales"),
        2
    )
).withColumn(
    "sales_difference",
    F.round(
        F.abs(
            F.col("Global_Sales") -
            F.col("regional_sales_total")
        ),
        2
    )
)

print("--- SALES RECONCILIATION ---")

sales_df.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(
            F.col("sales_difference") > 0,
            1
        ).otherwise(0)
    ).alias("records_with_difference"),
    F.max("sales_difference").alias("maximum_difference")
).show()

print("Largest sales differences:")

sales_df.orderBy(
    F.desc("sales_difference")
).select(
    "Rank",
    "Name",
    "Global_Sales",
    "regional_sales_total",
    "sales_difference"
).show(10, truncate=False)

print("ADDITIONAL PROFILING COMPLETED")


In [0]:

# =========================================================
# PHASE 3: SILVER LAYER
# DATA CLEANING AND STANDARDIZATION
# =========================================================

from pyspark.sql import functions as F

BRONZE_TABLE = (
    "workspace.video_game_sales.bronze_video_game_sales"
)

SILVER_TABLE = (
    "workspace.video_game_sales.silver_video_game_sales"
)

# 1. Read Bronze
bronze_df = spark.table(BRONZE_TABLE)

# 2. Preserve raw year and standardize column names
silver_df = bronze_df.select(
    F.col("Rank").alias("rank"),
    F.trim(F.col("Name")).alias("name"),
    F.trim(F.col("Platform")).alias("platform"),
    F.col("Year").alias("year_raw"),
    F.trim(F.col("Genre")).alias("genre"),
    F.trim(F.col("Publisher")).alias("publisher"),
    F.col("NA_Sales").cast("double").alias("na_sales"),
    F.col("EU_Sales").cast("double").alias("eu_sales"),
    F.col("JP_Sales").cast("double").alias("jp_sales"),
    F.col("Other_Sales").cast("double").alias("other_sales"),
    F.col("Global_Sales").cast("double").alias("global_sales"),
    F.col("source_file"),
    F.col("ingestion_timestamp")
)

# 3. Convert valid year values
silver_df = silver_df.withColumn(
    "year",
    F.when(
        F.col("year_raw").rlike(r"^\d{4}$"),
        F.col("year_raw").cast("int")
    )
)

# 4. Add year quality status
silver_df = silver_df.withColumn(
    "year_status",
    F.when(
        F.col("year").isNotNull(),
        F.lit("VALID")
    ).otherwise(
        F.lit("MISSING_OR_INVALID")
    )
)

# 5. Calculate regional sales total
silver_df = silver_df.withColumn(
    "regional_sales_total",
    F.round(
        F.coalesce(F.col("na_sales"), F.lit(0.0)) +
        F.coalesce(F.col("eu_sales"), F.lit(0.0)) +
        F.coalesce(F.col("jp_sales"), F.lit(0.0)) +
        F.coalesce(F.col("other_sales"), F.lit(0.0)),
        2
    )
)

# 6. Calculate reconciliation difference
silver_df = silver_df.withColumn(
    "sales_difference",
    F.round(
        F.abs(
            F.col("global_sales") -
            F.col("regional_sales_total")
        ),
        2
    )
)

# 7. Add reconciliation status
silver_df = silver_df.withColumn(
    "sales_reconciliation_status",
    F.when(
        F.col("sales_difference") == 0,
        F.lit("MATCH")
    ).otherwise(
        F.lit("DIFFERENCE_DETECTED")
    )
)

# 8. Write Silver Delta table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table created:", SILVER_TABLE)

# 9. Validate record preservation
silver_count = spark.table(SILVER_TABLE).count()
bronze_count = bronze_df.count()

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)

assert bronze_count == silver_count

print("SILVER RECORD RECONCILIATION PASSED")


In [0]:
silver_df = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

print("TOTAL SILVER RECORDS:", silver_df.count())

print("\n--- YEAR QUALITY ---")

silver_df.groupBy("year_status") \
    .count() \
    .show()

print("\n--- SALES RECONCILIATION ---")

silver_df.groupBy("sales_reconciliation_status") \
    .count() \
    .show()

print("\n--- NULL YEAR VALUES ---")

print(
    "Missing years:",
    silver_df.filter(F.col("year").isNull()).count()
)

print("\n--- SILVER SCHEMA ---")

silver_df.printSchema()

print("\n--- SAMPLE RECORDS ---")

display(silver_df.limit(10))